##Setup

You will need to make a copy of this notebook in your Google Drive before you can edit the homework files. You can do so with **File &rarr; Save a copy in Drive**.

In [ ]:
#@title mount your Google Drive
#@markdown Your work will be stored in a folder called `hw_16831` by default to prevent Colab instance timeouts from deleting your edits.

import os
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
#@title set up mount symlink

DRIVE_PATH = '/content/gdrive/My\ Drive/hw_16831'
DRIVE_PYTHON_PATH = DRIVE_PATH.replace('\\', '')
if not os.path.exists(DRIVE_PYTHON_PATH):
  %mkdir $DRIVE_PATH

## the space in `My Drive` causes some issues,
## make a symlink to avoid this
SYM_PATH = '/content/hw_16831'
if not os.path.exists(SYM_PATH):
  !ln -s $DRIVE_PATH $SYM_PATH

In [ ]:
#@title apt install requirements

#@markdown Run each section with Shift+Enter

#@markdown Double-click on section headers to show code.

!apt update 
!apt install -y --no-install-recommends \
        build-essential \
        curl \
        git \
        gnupg2 \
        make \
        cmake \
        ffmpeg \
        swig \
        libz-dev \
        unzip \
        zlib1g-dev \
        libffi-dev \
        libglfw3 \
        libglfw3-dev \
        libxrandr2 \
        libxinerama-dev \
        libxi6 \
        libxcursor-dev \
        libgl1-mesa-dev \
        libgl1-mesa-glx \
        libglew-dev \
        libosmesa6-dev \
        lsb-release \
        ack-grep \
        patchelf \
        wget \
        xpra \
        xserver-xorg-dev \
        xvfb \
        python3-opengl \
        software-properties-common \
        ffmpeg > /dev/null 2>&1

In [ ]:
#@title clone homework repo
#@markdown Clones 16831-S26-HW. If you have your own fork with your code, replace the URL with your repo.

%cd $SYM_PATH
!rm -rf 16831-S26-HW
!git clone https://github.com/cmuroboticsdrl/16831-S26-HW.git

In [ ]:
#@title install requirements

%cd $SYM_PATH/16831-S26-HW/hw3
%pip install -r requirements.txt -q
%pip install -e . -q

In [ ]:
!pip install numpy

In [ ]:
# downloads mujoco from source
!wget https://mujoco.org/download/mujoco210-linux-x86_64.tar.gz
!tar xzvf mujoco210-linux-x86_64.tar.gz
!mkdir -p ~/.mujoco
!mv mujoco210 ~/.mujoco/mujoco210
!rm mujoco*

%pip install -U mujoco
%pip install -U 'mujoco-py<2.2,>=2.1'
%pip install -U pyvirtualdisplay
%pip install -U gym-notebook-wrapper
%pip install -U "cython<3"

!cp /root/.mujoco/mujoco210/bin/*.so /usr/lib/x86_64-linux-gnu/

In [ ]:
# set env variables
import os
os.environ['LD_LIBRARY_PATH'] += ':/root/.mujoco/mujoco210/bin'
os.environ['MUJOCO_PY_MUJOCO_PATH'] = '/root/.mujoco/mujoco210'
os.environ['LD_LIBRARY_PATH'] += ':/usr/lib/nvidia'

In [ ]:
#@title set up virtual display

from pyvirtualdisplay import Display

display = Display(visible=0, size=(1400, 900))
display.start()

In [ ]:
#@title test virtual display

#@markdown If you see a video of a four-legged ant fumbling about, setup is complete!
import numpy as np
if not hasattr(np, "bool8"):
    np.bool8 = np.bool_

import gym
import gnwrapper

env = gnwrapper.LoopAnimation(gym.make('Ant-v4'))

observation = env.reset()
for i in range(100):
    obs, rew, term, _ = env.step(env.action_space.sample())
    env.render()
    if term:
      break

env.display()
env.close()

## Editing Code / Using your local code

To edit code in Colab, click the folder icon on the left and open files under `hw_16831/16831-S26-HW/hw3/rob831/`. To use code you wrote on your Mac: either (A) push to a GitHub fork and change the clone URL in the "clone homework repo" cell to your fork, or (B) after cloning, upload your `dqn_agent.py`, `dqn_critic.py`, `argmax_policy.py` into the corresponding `rob831/agents/`, `rob831/critics/`, `rob831/policies/` folders. Colab has a ~12 hour timeout; we sync to Drive so you can re-mount and re-install on a new instance.

## Run DQN and Double DQN

In [ ]:
#@title imports
import os
import time

from rob831.infrastructure.rl_trainer import RL_Trainer
from rob831.agents.dqn_agent import DQNAgent
from rob831.infrastructure.dqn_utils import get_env_kwargs

In [ ]:
#@title runtime arguments

class Args:

  def __getitem__(self, key):
    return getattr(self, key)

  def __setitem__(self, key, val):
    setattr(self, key, val)

  def __contains__(self, key):
    return hasattr(self, key)

  env_name = 'LunarLander-v3' #@param ['LunarLander-v3', 'PongNoFrameSkip-v4']
  exp_name = 'q3_dqn' #@param

  ## PDF will tell you how to set ep_len
  ## and discount for each environment
  ep_len = 200 #@param {type: "integer"}

  #@markdown batches and steps
  batch_size = 32 #@param {type: "integer"}
  eval_batch_size = 1000 #@param {type: "integer"}

  num_agent_train_steps_per_iter = 1 #@param {type: "integer"}

  num_critic_updates_per_agent_update = 1 #@param {type: "integer"}
  
  #@markdown Q-learning parameters
  double_q = False #@param {type: "boolean"}

  #@markdown system
  save_params = False #@param {type: "boolean"}
  no_gpu = True #@param {type: "boolean"}
  which_gpu = 0 #@param {type: "integer"}
  seed = 1 #@param {type: "integer"}

  #@markdown logging
  ## default is to not log video so
  ## that logs are small enough to be
  ## uploaded to gradscope
  video_log_freq =  -1 #@param {type: "integer"}
  scalar_log_freq =  10000#@param {type: "integer"}


args = Args()

## ensure compatibility with hw1 code
args['train_batch_size'] = args['batch_size']

if args['video_log_freq'] > 0:
  import warnings
  warnings.warn(
      '''\nLogging videos will make eventfiles too'''
      '''\nlarge for the autograder. Set video_log_freq = -1'''
      '''\nfor the runs you intend to submit.''')

In [ ]:
#@title create directories for logging

SYM_PATH = globals().get('SYM_PATH', '/content/hw_16831')
data_path = os.path.join(SYM_PATH, '16831-S26-HW', 'hw3', 'data')
if not (os.path.exists(data_path)):
    os.makedirs(data_path)

logdir = args.exp_name + '_' + args.env_name + '_' + time.strftime("%d-%m-%Y_%H-%M-%S")
logdir = os.path.join(data_path, logdir)
args['logdir'] = logdir
if not(os.path.exists(logdir)):
    os.makedirs(logdir)

print("LOGGING TO: ", logdir)


In [ ]:
#@title Define Q-function trainer

class Q_Trainer(object):

    def __init__(self, params):
        self.params = params

        train_args = {
            'num_agent_train_steps_per_iter': params['num_agent_train_steps_per_iter'],
            'num_critic_updates_per_agent_update': params['num_critic_updates_per_agent_update'],
            'train_batch_size': params['batch_size'],
            'double_q': params['double_q'],
        }

        env_args = get_env_kwargs(params['env_name'])

        for k, v in env_args.items():
          params[k] = v

        self.params['agent_class'] = DQNAgent
        self.params['agent_params'] = params
        self.params['train_batch_size'] = params['batch_size']
        self.params['env_wrappers'] = env_args['env_wrappers']

        self.rl_trainer = RL_Trainer(self.params)

    def run_training_loop(self):
        self.rl_trainer.run_training_loop(
            self.params['num_timesteps'],
            collect_policy = self.rl_trainer.agent.actor,
            eval_policy = self.rl_trainer.agent.actor,
            )

In [ ]:
#@title run training

trainer = Q_Trainer(args)
trainer.run_training_loop()

## Run all DQN experiments (optional)

Run the cell below to execute **DQN** and **Double DQN** on LunarLander in sequence. Logs go to `data/`. You can download the `data` folder from the file browser (under `hw_16831/16831-S26-HW/hw3/data`) or from Drive after the run.

In [ ]:
#@title run all DQN experiments (DQN + Double DQN on LunarLander)
SYM_PATH = globals().get('SYM_PATH', '/content/hw_16831')
data_path = os.path.join(SYM_PATH, '16831-S26-HW', 'hw3', 'data')
os.makedirs(data_path, exist_ok=True)

experiments = [
    {'exp_name': 'q3_dqn', 'double_q': False},
    {'exp_name': 'q4_double_dqn', 'double_q': True},
]
for cfg in experiments:
    print('\n' + '='*60)
    print(f"Running {cfg['exp_name']} (double_q={cfg['double_q']})")
    print('='*60)
    args = Args()
    args['exp_name'] = cfg['exp_name']
    args['double_q'] = cfg['double_q']
    args['env_name'] = 'LunarLander-v3'
    args['train_batch_size'] = args['batch_size']
    logdir = args['exp_name'] + '_' + args['env_name'] + '_' + time.strftime("%d-%m-%Y_%H-%M-%S")
    args['logdir'] = os.path.join(data_path, logdir)
    os.makedirs(args['logdir'], exist_ok=True)
    trainer = Q_Trainer(args)
    trainer.run_training_loop()

print('\nAll experiments done. Logs saved under:', data_path)

In [ ]:
#@markdown You can visualize your runs with tensorboard from within the notebook

## requires tensorflow==2.3.0
%load_ext tensorboard
%tensorboard --logdir /content/hw_16831/16831-S26-HW/hw3/data